In [0]:
%sql
-- Total/average valuation by district
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_valuation_by_district AS
SELECT 
dd.cd,
COUNT(*) AS total_permits,
SUM(valuation) AS total_valuation, 
ROUND(AVG(valuation),2) AS avg_valuation 
FROM la_lakehouse.gold.fact_permits AS fp
LEFT JOIN la_lakehouse.gold.dim_district dd
ON fp.district_key = dd.district_key
WHERE dd.cd IS NOT NULL
GROUP BY dd.cd
ORDER BY total_valuation DESC

In [0]:
%sql
-- Total/average valuation by zone
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_valuation_by_zone AS
SELECT 
dz.zone,
COUNT(*) AS total_permits,
SUM(valuation) AS total_valuation, 
ROUND(AVG(valuation),2) AS avg_valuation 
FROM la_lakehouse.gold.fact_permits AS fp
LEFT JOIN la_lakehouse.gold.dim_zone dz
ON fp.zone_key = dz.zone_key
WHERE dz.zone IS NOT NULL
GROUP BY dz.zone
ORDER BY total_valuation DESC

In [0]:
%sql
-- Total/average valuation by census tract
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_valuation_by_ct AS
SELECT 
dct.ct,
COUNT(*) AS total_permits,
SUM(valuation) AS total_valuation, 
ROUND(AVG(valuation),2) AS avg_valuation 
FROM la_lakehouse.gold.fact_permits AS fp
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct
ON fp.ct_key = dct.ct_key
WHERE dct.ct IS NOT NULL
GROUP BY dct.ct
ORDER BY total_valuation DESC

In [0]:
%sql
-- Total/average valuation by year
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_valuation_by_year AS
SELECT 
YEAR(status_date) AS year,
COUNT(*) AS total_permits,
SUM(valuation) AS total_valuation, 
ROUND(AVG(valuation),2) AS avg_valuation 
FROM la_lakehouse.gold.fact_permits AS fp
WHERE YEAR(status_date) IS NOT NULL
GROUP BY year
ORDER BY total_valuation DESC


In [0]:
%sql
-- Does valuation correlate with approval velocity?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_valuation_approval_velocity AS
WITH duration AS (
    SELECT 
    fp.valuation,
    DATEDIFF(id.full_date, sd.full_date) AS approval_days
    FROM la_lakehouse.gold.fact_permits AS fp
    INNER JOIN la_lakehouse.gold.dim_date AS sd
        ON fp.submitted_date_key = sd.date_key
    INNER JOIN la_lakehouse.gold.dim_date AS id
        ON fp.issue_date_key = id.date_key
    WHERE fp.valuation IS NOT NULL 
      AND fp.valuation > 0
      AND id.full_date >= sd.full_date
)
SELECT 
COUNT(*) AS total_permits,
ROUND(CORR(approval_days, valuation), 4) AS correlation_coeffcient,
ROUND(AVG(approval_days),2) AS avg_approval_days,
ROUND(AVG(valuation),2) AS avg_valuation
FROM duration;


In [0]:
%sql
--  Do permits in Hillside Ordinance areas show different approval times, valuation, or ADU activity than non-hillside permits? (uses dim_hillside_ordinance_area)
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_dim_hillside_ordinance_area AS
WITH base_permits AS (
    SELECT 
    CASE WHEN UPPER(hl) LIKE 'Y%' THEN 'Hillside' ELSE 'Non-Hillside' END AS hl,
    fp.valuation, 
    DATEDIFF(id.full_date, sd.full_date) AS approval_days,
    CASE WHEN dut.use_desc LIKE '%Accessory Dwelling Unit%' THEN 1 ELSE 0 END AS adu_flag
    FROM la_lakehouse.gold.fact_permits AS fp 
    LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS hoa
    ON fp.hl_key = hoa.hl_key
    LEFT JOIN la_lakehouse.gold.dim_use_type AS dut
    ON fp.use_type_key = dut.use_type_key
    INNER JOIN la_lakehouse.gold.dim_date AS sd 
    ON fp.submitted_date_key = sd.date_key
    INNER JOIN la_lakehouse.gold.dim_date AS id
    ON fp.issue_date_key = id.date_key
    WHERE fp.valuation IS NOT NULL
    AND sd.full_date IS NOT NULL
    AND id.full_date IS NOT NULL
    AND id.full_date >= sd.full_date
)
SELECT 
hl,
COUNT(*) AS total_permits,
ROUND(AVG(approval_days),2) AS avg_approval_speed,
ROUND(AVG(valuation),2) AS avg_valuation,
SUM(valuation) AS total_valuation,
SUM(adu_flag) AS adu_count,
ROUND(100.0 * (SUM(adu_flag)/COUNT(*)),2) AS adu_percentage
FROM base_permits
GROUP BY hl

In [0]:
%sql 
-- Which certified neighborhood councils have the highest permit activity and average valuation within their boundaries? (uses dim_certified_neighborhood_council)
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_cnc_valuation_activity AS
SELECT 
    cnc.cnc,
    COUNT(*) AS total_permits,
    ROUND(AVG(fp.valuation), 2) AS avg_valuation,
    SUM(fp.valuation) AS total_invested
FROM la_lakehouse.gold.fact_permits AS fp
INNER JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS cnc
    ON fp.cnc_key = cnc.cnc_key
WHERE fp.valuation IS NOT NULL
  AND cnc.cnc IS NOT NULL
GROUP BY 1
ORDER BY total_permits DESC;

#Testing Tables

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_valuation_by_district; 

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_valuation_by_zone;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_valuation_by_ct;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_valuation_by_year

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_valuation_approval_velocity;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_dim_hillside_ordinance_area

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_cnc_valuation_activity;